# E2 — Kết quả từng phần, chưa hoàn thành thiết kế

Chỉ 6/12 run SUCCESS. A1/A2 có seeds 7/42; A3/A4 chỉ seed 7. Không điền điểm giả hoặc coi missing runs đã hoàn thành.


In [ ]:
import os, sys, json
from pathlib import Path
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src/analysis/master_session1.py").is_file()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "scripts"))
from src.analysis.master_session1 import ARCHIVE_NAME, ARCHIVE_SHA256, BUNDLE_HASH, analyze
TABLES = ROOT / "results/tables/master_session1"
FULL_AUDIT = False  # True needs the private original ZIP and matching dataset; never trains.
REBUILD_FIGURES = False
if FULL_AUDIT:
    archive = Path(os.environ.get("SEQREC_MASTER_ARCHIVE", str(ROOT / "artifacts/master_session1" / ARCHIVE_NAME)))
    bundle = Path(os.environ.get("SEQREC_BUNDLE", str(ROOT / "data/processed/beauty" / BUNDLE_HASH)))
    analyze(archive, ROOT, bundle, resamples=10000)
audit = json.loads((TABLES / "artifact_audit.json").read_text())
assert audit["state"] == "VERIFIED" and audit["archive_sha256"] == ARCHIVE_SHA256
assert audit["E1_success"] == 16 and audit["E2_success"] == 6 and audit["E7_gpu_state"] == "DEFERRED"
print("Source SHA256:", ARCHIVE_SHA256)
print("Mode:", "raw ZIP re-audited" if FULL_AUDIT else "versioned audited aggregate tables; not a fresh raw re-audit")


In [ ]:
progress = pd.read_csv(TABLES / "original_progress.csv")
display(progress[progress.experiment == "E2"])
metrics = pd.read_csv(TABLES / "per_seed_metrics.csv")
display(metrics[metrics.experiment == "E2"])
summary = pd.read_csv(TABLES / "metric_summary.csv")
display(summary[summary.experiment == "E2"])


## Chỉ so sánh matched completed seeds

A2−A1 dùng seeds 7/42; A4−A3 chỉ seed 7. CI không thay thế sáu run thiếu. Giữ nguyên matched-capacity backbone, hyperparameters và quy tắc checkpoint validation.


In [ ]:
paired = pd.read_csv(TABLES / "paired_bootstrap.csv")
display(paired[paired.experiment == "E2"])
display(pd.read_csv(TABLES / "course_manifest.csv").query("experiment == 'E2'"))


In [ ]:
from gen_fig_master_session1 import generate
if REBUILD_FIGURES:
    generate(ROOT)
display(Image(filename=str(ROOT / "results/figures/master_session1/e2_partial_results.png")))


## Báo cáo và trạng thái

Đọc [báo cáo tiếng Việt](../docs/MASTER_SESSION1_ANALYSIS_VI.md). Phạm vi thu gọn do tài nguyên GPU và yêu cầu môn học, không theo mô hình có test score cao. E1 complete, E2 partial, E3 pending/reuse available, E4/E5 deferred; TIGER/CRAMER theory only. Không tuyên bố các phase chưa chạy đã hoàn tất.
